In [2]:
%run "/Users/manojrammopati/Public/Projects/bupa_insurance_project/_01_Bronze/Jupyter Notebooks/00_bronze_data_connector.ipynb"


25/12/11 18:51:32 WARN Utils: Your hostname, Manojrams-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 192.168.0.219 instead (on interface en0)
25/12/11 18:51:32 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/opt/anaconda3/envs/spark_local/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /Users/manojrammopati/.ivy2/cache
The jars for the packages stored in: /Users/manojrammopati/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.hadoop#hadoop-azure added as a dependency
com.azure#azure-storage-blob added as a dependency
com.azure#azure-identity added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-100522c8-64c0-44da-aa31-2a67d2dc31f1;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.1.0 in central
	found io.delta#delta-storage;3.1.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.apache.hadoop#hadoop-azure;3.3.4 in central
	found org.apache.httpcomponents#httpclient;4.5.13 in central
	found org.apache.httpcomponents#httpcore;4.4.13 in central
	found commons-logging#commons-logging;1.1.3 in central
	found commons-codec#commons-codec;1.15 in central
	found com.microsoft.azure#azure-storage;7.0.1 in central
	found com.fasterxml.jackson.core#jackson-core;2.12.7 in c

Preconnectors ran successfully and connected to 'clientdatastorage' storage accounts
 'policy_df, claim_df, member_df, provider_df' are ready to use from container 'raw data'(meridian architecture of bronze) 


In [27]:



DB_GOLD = "bupa_gold"

spark.sql(f"CREATE DATABASE IF NOT EXISTS {DB_GOLD}")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {DB_GOLD}.dm_policy_retention
    USING DELTA
    LOCATION 'abfss://golddata@clientdatastorage.dfs.core.windows.net/dm_policy_retention'
""")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {DB_GOLD}.fact_policies
    USING DELTA
    LOCATION 'abfss://golddata@clientdatastorage.dfs.core.windows.net/fact_policies'
""")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {DB_GOLD}.dm_claims_experience
    USING DELTA
    LOCATION 'abfss://golddata@clientdatastorage.dfs.core.windows.net/dm_claims_experience'
""")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {DB_GOLD}.star_members
    USING DELTA
    LOCATION 'abfss://golddata@clientdatastorage.dfs.core.windows.net/star_members'
""")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {DB_GOLD}.dm_member_value
    USING DELTA
    LOCATION 'abfss://golddata@clientdatastorage.dfs.core.windows.net/dm_member_value'
""")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {DB_GOLD}.ft_policy_churn_split
    USING DELTA
    LOCATION 'abfss://golddata@clientdatastorage.dfs.core.windows.net/ft_policy_churn_split'
""")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {DB_GOLD}.ft_claims_risk_split
    USING DELTA
    LOCATION 'abfss://golddata@clientdatastorage.dfs.core.windows.net/ft_claims_risk_split'
""")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {DB_GOLD}.scored_policy_churn
    USING DELTA
    LOCATION 'abfss://golddata@clientdatastorage.dfs.core.windows.net/scored_policy_churn'
""")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {DB_GOLD}.vw_scored_policy_churn
    USING DELTA
    LOCATION 'abfss://golddata@clientdatastorage.dfs.core.windows.net/scored_policy_churn'
""")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {DB_GOLD}.scored_claims_fraud
    USING DELTA
    LOCATION 'abfss://golddata@clientdatastorage.dfs.core.windows.net/scored_claims/claims_fraud'
""")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {DB_GOLD}.vw_scored_claims_fraud
    USING DELTA
    LOCATION 'abfss://golddata@clientdatastorage.dfs.core.windows.net/vw_scored_claims_fraud'
""")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {DB_GOLD}.scored_claims_high_cost
    USING DELTA
    LOCATION 'abfss://golddata@clientdatastorage.dfs.core.windows.net/scored_claims/high_cost_claims'
""")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {DB_GOLD}.vw_scored_claims_high_cost
    USING DELTA
    LOCATION 'abfss://golddata@clientdatastorage.dfs.core.windows.net/vw_scored_claims_high_cost'
""")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {DB_GOLD}.star_policies
    USING DELTA
    LOCATION 'abfss://golddata@clientdatastorage.dfs.core.windows.net/star_policies'
""")




DataFrame[]

# Cell 1 – Setup

In [ ]:
# Dashboard Views – Setup

from pyspark.sql import functions as F

DB_GOLD = "bupa_gold"

print("Using Gold database:", DB_GOLD)
spark.sql(f"USE {DB_GOLD}")
spark.sql("SHOW TABLES").show(truncate=False)


Using Gold database: bupa_gold
+---------+--------------------+-----------+
|namespace|tableName           |isTemporary|
+---------+--------------------+-----------+
|bupa_gold|dm_claims_experience|false      |
|bupa_gold|dm_policy_retention |false      |
|bupa_gold|fact_policies       |false      |
|bupa_gold|vw_policy_portfolio |false      |
+---------+--------------------+-----------+



# Cell 2 – Policy portfolio view (detail from fact_policies)

In [ ]:
# Cell 2 — Policy portfolio view (detail-level, 1 row per policy)

spark.sql(f"""
CREATE OR REPLACE VIEW {DB_GOLD}.vw_policy_portfolio AS
SELECT
    Policy_ID,
    Customer_ID,
    Product_Line,
    Channel,
    Sum_Insured_GBP,
    Annual_Premium_GBP,
    Policy_Start_Date,
    Policy_End_Date,
    Policy_Duration_Days,
    Tenure_Band,
    Premium_Band,
    Discount_Band,
    Renewal_Offered_Flag,
    Renewal_Accepted_Flag,
    Renewal_Conversion,
    Renewal_Outcome,
    dq_money_valid,
    dq_discount_valid,
    dq_renewal_valid,
    created_at,
    source_system,
    record_hash
FROM {DB_GOLD}.fact_policies
""")

print("✅ Created view:", f"{DB_GOLD}.vw_policy_portfolio")
spark.table(f"{DB_GOLD}.vw_policy_portfolio").show(10, truncate=False)
spark.table(f"{DB_GOLD}.vw_policy_portfolio").printSchema()


✅ Created view: bupa_gold.vw_policy_portfolio


+----------+------------+------------+-------+------------------+------------------+-----------------+---------------+--------------------+-----------+-----------------+-------------+--------------------+---------------------+------------------+---------------+--------------+-----------------+----------------+--------------------------+----------------------------+----------------------------------------------------------------+
|Policy_ID |Customer_ID |Product_Line|Channel|Sum_Insured_GBP   |Annual_Premium_GBP|Policy_Start_Date|Policy_End_Date|Policy_Duration_Days|Tenure_Band|Premium_Band     |Discount_Band|Renewal_Offered_Flag|Renewal_Accepted_Flag|Renewal_Conversion|Renewal_Outcome|dq_money_valid|dq_discount_valid|dq_renewal_valid|created_at                |source_system               |record_hash                                                     |
+----------+------------+------------+-------+------------------+------------------+-----------------+---------------+----------------

Usage in Power BI / Tableau

Use this view for portfolio-level dashboards:

Premium by product/channel

Tenure distribution

Renewal outcome breakdown

Discount vs renewal conversion, etc.

# Cell 3 – Policy retention KPI view (from dm_policy_retention)

In [ ]:
# Cell 3 — Policy retention KPI view (aggregated)

spark.sql(f"""
CREATE OR REPLACE VIEW {DB_GOLD}.vw_policy_retention_kpi AS
SELECT
    Product_Line,
    Channel,
    Tenure_Band,
    Policy_Start_Year,
    Policy_Count,
    Total_Premium_GBP,
    Avg_Premium_GBP,
    Offer_Rate,
    Acceptance_Rate,
    Clean_Renewal_Conversion
FROM {DB_GOLD}.dm_policy_retention
""")

print("✅ Created view:", f"{DB_GOLD}.vw_policy_retention_kpi")
spark.table(f"{DB_GOLD}.vw_policy_retention_kpi").show(10, truncate=False)
spark.table(f"{DB_GOLD}.vw_policy_retention_kpi").printSchema()


✅ Created view: bupa_gold.vw_policy_retention_kpi
+------------+-------+-----------+-----------------+------------+-----------------+---------------+----------+---------------+------------------------+
|Product_Line|Channel|Tenure_Band|Policy_Start_Year|Policy_Count|Total_Premium_GBP|Avg_Premium_GBP|Offer_Rate|Acceptance_Rate|Clean_Renewal_Conversion|
+------------+-------+-----------+-----------------+------------+-----------------+---------------+----------+---------------+------------------------+
|Dental      |Broker |6–12 months|2024             |10          |216588.0         |21658.8        |1.0       |0.7            |0.7                     |
|Dental      |Partner|1–2 years  |2020             |832         |2.5988647E7      |31236.35       |0.899     |0.6238         |0.6939                  |
|Dental      |Online |6–12 months|2023             |25          |65750.0          |2630.0         |1.0       |0.84           |0.84                    |
|Dental      |Partner|6–12 months|2021

Use this for:

High-level KPI tiles and trend charts
(e.g. acceptance rate by year, conversion by channel & tenure band).

# Cell 4 – Member profile & value views

In [ ]:
# Cell 4 — Member profile (star_members) and member value (dm_member_value)

# 4a) Member profile – detailed, 1 row per member
spark.sql(f"""
CREATE OR REPLACE VIEW {DB_GOLD}.vw_member_profile AS
SELECT
    Member_ID,
    Policy_ID,
    Age,
    Gender,
    BMI,
    Chronic_Disease,
    Employment_Status,
    Region_Code,
    dq_age_valid,
    dq_bmi_valid
FROM {DB_GOLD}.star_members
""")

print("✅ Created view:", f"{DB_GOLD}.vw_member_profile")
spark.table(f"{DB_GOLD}.vw_member_profile").show(10, truncate=False)
spark.table(f"{DB_GOLD}.vw_member_profile").printSchema()


# 4b) Member value – aggregated per segment, etc. (from dm_member_value)

spark.sql(f"""
CREATE OR REPLACE VIEW {DB_GOLD}.vw_member_value_kpi AS
SELECT
    *
FROM {DB_GOLD}.dm_member_value
""")

print("✅ Created view:", f"{DB_GOLD}.vw_member_value_kpi")
spark.table(f"{DB_GOLD}.vw_member_value_kpi").show(10, truncate=False)
spark.table(f"{DB_GOLD}.vw_member_value_kpi").printSchema()


✅ Created view: bupa_gold.vw_member_profile


+------------+----------+---+------+------------------+---------------+-----------------+-----------+------------+------------+
|Member_ID   |Policy_ID |Age|Gender|BMI               |Chronic_Disease|Employment_Status|Region_Code|dq_age_valid|dq_bmi_valid|
+------------+----------+---+------+------------------+---------------+-----------------+-----------+------------+------------+
|MEM_00000008|P_00000008|56 |F     |23.140940890221643|Asthma         |Student          |280        |1           |1           |
|MEM_00001714|P_00001714|71 |M     |27.963719840043765|None           |Employed         |360        |1           |1           |
|MEM_00001746|P_00001746|75 |M     |30.541100957674118|Diabetes       |Student          |80         |1           |1           |
|MEM_00001967|P_00001967|37 |F     |25.982850362358867|Hypertension   |Employed         |280        |1           |1           |
|MEM_00001995|P_00001995|66 |M     |32.6294573451364  |Asthma         |Employed         |360        |1  

+----------+------------+---+------+------------------+-----------+---------------+-----------------+------+--------+----------+-------------+------------+-------------+----------------+--------------+------------+------------+------------+------------+-------+------------------+------------------+-----------------+---------------+--------------------+--------------------+---------------------+------------------+-----------+-----------------+-------------+---------------+--------------+-----------------+----------------+--------------------------+----------------------------+----------------------------------------------------------------+------------+----------------+----------------------+---------------+---------------+--------------------+-----------------+---------------+--------+
|Policy_ID |Member_ID   |Age|Gender|BMI               |Smoker_Flag|Chronic_Disease|Employment_Status|Region|Age_Band|BMI_Band  |Smoker_Status|Chronic_Flag|Chronic_Group|Employment_Group|Region_Group  |dq

Use:

vw_member_profile → demographic / health profile dashboards

vw_member_value_kpi → segment-level LTV / claim cost / premium KPIs.

# Cell 5 – Claims experience view (from dm_claims_experience)

In [ ]:
# Cell 5 — Claims experience KPIs (aggregated)

spark.sql(f"""
CREATE OR REPLACE VIEW {DB_GOLD}.vw_claims_experience_kpi AS
SELECT
    *
FROM {DB_GOLD}.dm_claims_experience
""")

print("✅ Created view:", f"{DB_GOLD}.vw_claims_experience_kpi")
spark.table(f"{DB_GOLD}.vw_claims_experience_kpi").show(10, truncate=False)
spark.table(f"{DB_GOLD}.vw_claims_experience_kpi").printSchema()


✅ Created view: bupa_gold.vw_claims_experience_kpi


+---------+-----------+----------+-------------+------------+----------+------------------+-----------+----------+------------+-------------------+--------------+----------------------+--------------------+--------------+-------------+------------------+------------------+--------------+---------+-------------------+----------------+-----------------------+
|Claim_ID |Provider_ID|Member_Key|Date_Reported|Date_Settled|Payout_GBP|Claim_Amount_GBP  |Fraud_Label|Claim_Type|Claim_Status|Provider_Fraud_Flag|Days_To_Settle|Payout_to_Amount_Ratio|High_Cost_Claim_Flag|dq_money_valid|dq_date_valid|Exposure_GBP      |Payout_Ratio      |High_Cost_Flag|Cost_Band|Settlement_SLA_Band|Open_Closed_Flag|Provider_Fraud_Flag_Ref|
+---------+-----------+----------+-------------+------------+----------+------------------+-----------+----------+------------+-------------------+--------------+----------------------+--------------------+--------------+-------------+------------------+------------------+-------

Use this for:

Loss ratios by claim type / status / provider risk tier

High-cost vs normal claim mix

Fraud-labelled claim rates, etc.

# Cell 6 – ML: policy churn feature view

This doesn’t depend on the trained model; it just exposes the ML feature table, which is exactly what a data scientist or ML Ops engineer would connect to.

In [ ]:
# Cell 6 — ML policy churn feature view (from ft_policy_churn_split)

spark.sql(f"""
CREATE OR REPLACE VIEW {DB_GOLD}.vw_ml_policy_churn_features AS
SELECT
    Policy_ID,
    Customer_ID,
    Product_Line,
    Channel,
    Sum_Insured_GBP,
    Annual_Premium_GBP,
    Policy_Start_Date,
    Policy_End_Date,
    Policy_Duration_Days,
    Renewal_Offered_Flag,
    Renewal_Accepted_Flag,
    Renewal_Conversion,
    Tenure_Band,
    Premium_Band,
    Discount_Band,
    dq_money_valid,
    dq_discount_valid,
    dq_renewal_valid,
    Churn_Label,
    Is_Discounted,
    Premium_per_1k_SumInsured,
    dataset_split
FROM {DB_GOLD}.ft_policy_churn_split
""")

print("✅ Created view:", f"{DB_GOLD}.vw_ml_policy_churn_features")
spark.table(f"{DB_GOLD}.vw_ml_policy_churn_features").groupBy("dataset_split", "Churn_Label").count().show()
spark.table(f"{DB_GOLD}.vw_ml_policy_churn_features").show(10, truncate=False)


✅ Created view: bupa_gold.vw_ml_policy_churn_features


+-------------+-----------+------+
|dataset_split|Churn_Label| count|
+-------------+-----------+------+
|        train|          0|159495|
|         test|          0| 39820|
|        train|          1| 68818|
|        train|       NULL| 76429|
|         test|          1| 17306|
|         test|       NULL| 19241|
+-------------+-----------+------+



+----------+------------+------------+-------+------------------+------------------+-----------------+---------------+--------------------+--------------------+---------------------+------------------+-----------+-----------------+-------------+--------------+-----------------+----------------+-----------+-------------+-------------------------+-------------+
|Policy_ID |Customer_ID |Product_Line|Channel|Sum_Insured_GBP   |Annual_Premium_GBP|Policy_Start_Date|Policy_End_Date|Policy_Duration_Days|Renewal_Offered_Flag|Renewal_Accepted_Flag|Renewal_Conversion|Tenure_Band|Premium_Band     |Discount_Band|dq_money_valid|dq_discount_valid|dq_renewal_valid|Churn_Label|Is_Discounted|Premium_per_1k_SumInsured|dataset_split|
+----------+------------+------------+-------+------------------+------------------+-----------------+---------------+--------------------+--------------------+---------------------+------------------+-----------+-----------------+-------------+--------------+----------------

Use for:

Monitoring label balance train vs test

Model input distributions (premium, tenure, discount behaviour)

Drift checks later.

# Cell 7 – ML: claims risk feature view

In [ ]:
# Cell 7 — ML claims risk feature view (from ft_claims_risk_split)

spark.sql(f"""
CREATE OR REPLACE VIEW {DB_GOLD}.vw_ml_claims_risk_features AS
SELECT
    Claim_ID,
    Member_Key,
    Provider_ID,
    Claim_Type_Name,
    Claim_Type_Code,
    Claim_Status,
    Claim_Amount_GBP,
    Payout_GBP,
    Payout_to_Amount_Ratio,
    High_Cost_Claim_Flag,
    Claim_Fraud_Label,
    Provider_Fraud_Label,
    Provider_Risk_Tier,
    dq_money_valid,
    dq_date_valid,
    Is_Fraudulent_Claim,
    Is_High_Cost,
    dataset_split
FROM {DB_GOLD}.ft_claims_risk_split
""")

print("✅ Created view:", f"{DB_GOLD}.vw_ml_claims_risk_features")
spark.table(f"{DB_GOLD}.vw_ml_claims_risk_features").groupBy("dataset_split", "Is_Fraudulent_Claim").count().show()
spark.table(f"{DB_GOLD}.vw_ml_claims_risk_features").show(10, truncate=False)


✅ Created view: bupa_gold.vw_ml_claims_risk_features


+-------------+-------------------+------+
|dataset_split|Is_Fraudulent_Claim| count|
+-------------+-------------------+------+
|        train|                  0|283635|
|         test|                  0| 71276|
|        train|                  1|162467|
|         test|                  1| 40833|
+-------------+-------------------+------+



+---------+----------+-----------+---------------+---------------+------------+------------------+----------+----------------------+--------------------+-----------------+--------------------+------------------+--------------+-------------+-------------------+------------+-------------+
|Claim_ID |Member_Key|Provider_ID|Claim_Type_Name|Claim_Type_Code|Claim_Status|Claim_Amount_GBP  |Payout_GBP|Payout_to_Amount_Ratio|High_Cost_Claim_Flag|Claim_Fraud_Label|Provider_Fraud_Label|Provider_Risk_Tier|dq_money_valid|dq_date_valid|Is_Fraudulent_Claim|Is_High_Cost|dataset_split|
+---------+----------+-----------+---------------+---------------+------------+------------------+----------+----------------------+--------------------+-----------------+--------------------+------------------+--------------+-------------+-------------------+------------+-------------+
|CLM110013|BENE15435 |PRV51790   |Travel         |TRAVEL         |Settled     |244.07511199775644|200.0     |0.8194198841618822    |0   

Use for:

Monitoring class balance (fraud / high-cost) train vs test

Feature distribution by claim type / provider risk tier.

In [ ]:
from pyspark.sql import functions as F

DB_GOLD = "bupa_gold"
spark.sql(f"USE {DB_GOLD}")

# ADLS config
GOLD_ACCOUNT   = "clientdatastorage"
GOLD_CONTAINER = "golddata"

def gold_path(name: str) -> str:
    return f"abfss://{GOLD_CONTAINER}@{GOLD_ACCOUNT}.dfs.core.windows.net/{name}"

paths_views = {
    "vw_policy_portfolio"         : gold_path("vw_policy_portfolio"),
    "vw_policy_retention_kpi"     : gold_path("vw_policy_retention_kpi"),
    "vw_member_profile"           : gold_path("vw_member_profile"),
    "vw_member_value_kpi"         : gold_path("vw_member_value_kpi"),
    "vw_claims_experience_kpi"    : gold_path("vw_claims_experience_kpi"),
    "vw_ml_policy_churn_features" : gold_path("vw_ml_policy_churn_features"),
    "vw_ml_claims_risk_features"  : gold_path("vw_ml_claims_risk_features"),
}

print("Materialisation targets:")
for k, v in paths_views.items():
    print(f"  {k:30s} -> {v}")


Materialisation targets:
  vw_policy_portfolio            -> abfss://golddata@clientdatastorage.dfs.core.windows.net/vw_policy_portfolio
  vw_policy_retention_kpi        -> abfss://golddata@clientdatastorage.dfs.core.windows.net/vw_policy_retention_kpi
  vw_member_profile              -> abfss://golddata@clientdatastorage.dfs.core.windows.net/vw_member_profile
  vw_member_value_kpi            -> abfss://golddata@clientdatastorage.dfs.core.windows.net/vw_member_value_kpi
  vw_claims_experience_kpi       -> abfss://golddata@clientdatastorage.dfs.core.windows.net/vw_claims_experience_kpi
  vw_ml_policy_churn_features    -> abfss://golddata@clientdatastorage.dfs.core.windows.net/vw_ml_policy_churn_features
  vw_ml_claims_risk_features     -> abfss://golddata@clientdatastorage.dfs.core.windows.net/vw_ml_claims_risk_features


# Cell 2 – Helper to write a view to ADLS

In [ ]:
def materialise_view(view_name: str, path: str):
    """
    Read a Spark SQL view from bupa_gold and write it as a Delta dataset to ADLS.
    Overwrites if it already exists.
    """
    print(f"\n▶ Materialising view {DB_GOLD}.{view_name} → {path}")
    
    df = spark.table(f"{DB_GOLD}.{view_name}")
    print("  Rowcount:", df.count())
    
    (df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .save(path))
    
    print("  ✅ Written as Delta:", path)


# Cell 3 – Materialise all 7 views

In [ ]:
# Policy views
materialise_view("vw_policy_portfolio",         paths_views["vw_policy_portfolio"])
materialise_view("vw_policy_retention_kpi",     paths_views["vw_policy_retention_kpi"])

# Member views
materialise_view("vw_member_profile",           paths_views["vw_member_profile"])
materialise_view("vw_member_value_kpi",         paths_views["vw_member_value_kpi"])

# Claims views
materialise_view("vw_claims_experience_kpi",    paths_views["vw_claims_experience_kpi"])

# ML feature views
materialise_view("vw_ml_policy_churn_features", paths_views["vw_ml_policy_churn_features"])
materialise_view("vw_ml_claims_risk_features",  paths_views["vw_ml_claims_risk_features"])



▶ Materialising view bupa_gold.vw_policy_portfolio → abfss://golddata@clientdatastorage.dfs.core.windows.net/vw_policy_portfolio
  Rowcount: 381109


25/12/07 23:33:32 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


  ✅ Written as Delta: abfss://golddata@clientdatastorage.dfs.core.windows.net/vw_policy_portfolio

▶ Materialising view bupa_gold.vw_policy_retention_kpi → abfss://golddata@clientdatastorage.dfs.core.windows.net/vw_policy_retention_kpi
  Rowcount: 183


  ✅ Written as Delta: abfss://golddata@clientdatastorage.dfs.core.windows.net/vw_policy_retention_kpi

▶ Materialising view bupa_gold.vw_member_profile → abfss://golddata@clientdatastorage.dfs.core.windows.net/vw_member_profile
  Rowcount: 381109


  ✅ Written as Delta: abfss://golddata@clientdatastorage.dfs.core.windows.net/vw_member_profile

▶ Materialising view bupa_gold.vw_member_value_kpi → abfss://golddata@clientdatastorage.dfs.core.windows.net/vw_member_value_kpi
  Rowcount: 381109


...

  ✅ Written as Delta: abfss://golddata@clientdatastorage.dfs.core.windows.net/vw_member_value_kpi

▶ Materialising view bupa_gold.vw_claims_experience_kpi → abfss://golddata@clientdatastorage.dfs.core.windows.net/vw_claims_experience_kpi
  Rowcount: 558211


  ✅ Written as Delta: abfss://golddata@clientdatastorage.dfs.core.windows.net/vw_claims_experience_kpi

▶ Materialising view bupa_gold.vw_ml_policy_churn_features → abfss://golddata@clientdatastorage.dfs.core.windows.net/vw_ml_policy_churn_features
  Rowcount: 381109


25/12/07 23:34:21 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


  ✅ Written as Delta: abfss://golddata@clientdatastorage.dfs.core.windows.net/vw_ml_policy_churn_features

▶ Materialising view bupa_gold.vw_ml_claims_risk_features → abfss://golddata@clientdatastorage.dfs.core.windows.net/vw_ml_claims_risk_features
  Rowcount: 558211


  ✅ Written as Delta: abfss://golddata@clientdatastorage.dfs.core.windows.net/vw_ml_claims_risk_features


You should see prints like:

Rowcount: ...

✅ Written as Delta: abfss://golddata@clientdatastorage.dfs.core.windows.net/vw_policy_portfolio

etc.

# (Optional) Cell 4 – Register external tables on top of ADLS paths

If you also want physical tables on top of those paths (not just views), you can add:

In [ ]:
for view_name, path in paths_views.items():
    table_name = f"{view_name}_tbl"   # e.g. vw_policy_portfolio_tbl
    full_table = f"{DB_GOLD}.{table_name}"
    
    print(f"\n▶ Registering external table {full_table} on {path}")
    
    spark.sql(f"DROP TABLE IF EXISTS {full_table}")
    spark.sql(f"""
    CREATE TABLE {full_table}
    USING DELTA
    LOCATION '{path}'
    """)
    
    print("  ✅ Registered:", full_table)
    spark.table(full_table).show(3, truncate=False)



▶ Registering external table bupa_gold.vw_policy_portfolio_tbl on abfss://golddata@clientdatastorage.dfs.core.windows.net/vw_policy_portfolio
  ✅ Registered: bupa_gold.vw_policy_portfolio_tbl


+----------+------------+------------+-------+------------------+------------------+-----------------+---------------+--------------------+-----------+-----------------+-------------+--------------------+---------------------+------------------+---------------+--------------+-----------------+----------------+--------------------------+----------------------------+----------------------------------------------------------------+
|Policy_ID |Customer_ID |Product_Line|Channel|Sum_Insured_GBP   |Annual_Premium_GBP|Policy_Start_Date|Policy_End_Date|Policy_Duration_Days|Tenure_Band|Premium_Band     |Discount_Band|Renewal_Offered_Flag|Renewal_Accepted_Flag|Renewal_Conversion|Renewal_Outcome|dq_money_valid|dq_discount_valid|dq_renewal_valid|created_at                |source_system               |record_hash                                                     |
+----------+------------+------------+-------+------------------+------------------+-----------------+---------------+----------------

+------------+----------+---+------+------------------+---------------+-----------------+-----------+------------+------------+
|Member_ID   |Policy_ID |Age|Gender|BMI               |Chronic_Disease|Employment_Status|Region_Code|dq_age_valid|dq_bmi_valid|
+------------+----------+---+------+------------------+---------------+-----------------+-----------+------------+------------+
|MEM_00000008|P_00000008|56 |F     |23.140940890221643|Asthma         |Student          |280        |1           |1           |
|MEM_00001714|P_00001714|71 |M     |27.963719840043765|None           |Employed         |360        |1           |1           |
|MEM_00001746|P_00001746|75 |M     |30.541100957674118|Diabetes       |Student          |80         |1           |1           |
+------------+----------+---+------+------------------+---------------+-----------------+-----------+------------+------------+
only showing top 3 rows


▶ Registering external table bupa_gold.vw_member_value_kpi_tbl on abfss://gold

+----------+------------+---+------+------------------+-----------+---------------+-----------------+------+--------+----------+-------------+------------+-------------+----------------+--------------+------------+------------+------------+------------+-------+------------------+------------------+-----------------+---------------+--------------------+--------------------+---------------------+------------------+-----------+-----------------+-------------+---------------+--------------+-----------------+----------------+--------------------------+----------------------------+----------------------------------------------------------------+------------+----------------+----------------------+---------------+---------------+--------------------+-----------------+---------------+--------+
|Policy_ID |Member_ID   |Age|Gender|BMI               |Smoker_Flag|Chronic_Disease|Employment_Status|Region|Age_Band|BMI_Band  |Smoker_Status|Chronic_Flag|Chronic_Group|Employment_Group|Region_Group  |dq

+---------+-----------+----------+-------------+------------+----------+------------------+-----------+----------+------------+-------------------+--------------+----------------------+--------------------+--------------+-------------+------------------+------------------+--------------+---------+-------------------+----------------+-----------------------+
|Claim_ID |Provider_ID|Member_Key|Date_Reported|Date_Settled|Payout_GBP|Claim_Amount_GBP  |Fraud_Label|Claim_Type|Claim_Status|Provider_Fraud_Flag|Days_To_Settle|Payout_to_Amount_Ratio|High_Cost_Claim_Flag|dq_money_valid|dq_date_valid|Exposure_GBP      |Payout_Ratio      |High_Cost_Flag|Cost_Band|Settlement_SLA_Band|Open_Closed_Flag|Provider_Fraud_Flag_Ref|
+---------+-----------+----------+-------------+------------+----------+------------------+-----------+----------+------------+-------------------+--------------+----------------------+--------------------+--------------+-------------+------------------+------------------+-------

+----------+------------+------------+-------+------------------+------------------+-----------------+---------------+--------------------+--------------------+---------------------+------------------+-----------+-----------------+-------------+--------------+-----------------+----------------+-----------+-------------+-------------------------+-------------+
|Policy_ID |Customer_ID |Product_Line|Channel|Sum_Insured_GBP   |Annual_Premium_GBP|Policy_Start_Date|Policy_End_Date|Policy_Duration_Days|Renewal_Offered_Flag|Renewal_Accepted_Flag|Renewal_Conversion|Tenure_Band|Premium_Band     |Discount_Band|dq_money_valid|dq_discount_valid|dq_renewal_valid|Churn_Label|Is_Discounted|Premium_per_1k_SumInsured|dataset_split|
+----------+------------+------------+-------+------------------+------------------+-----------------+---------------+--------------------+--------------------+---------------------+------------------+-----------+-----------------+-------------+--------------+----------------

+---------+----------+-----------+---------------+---------------+------------+------------------+----------+----------------------+--------------------+-----------------+--------------------+------------------+--------------+-------------+-------------------+------------+-------------+
|Claim_ID |Member_Key|Provider_ID|Claim_Type_Name|Claim_Type_Code|Claim_Status|Claim_Amount_GBP  |Payout_GBP|Payout_to_Amount_Ratio|High_Cost_Claim_Flag|Claim_Fraud_Label|Provider_Fraud_Label|Provider_Risk_Tier|dq_money_valid|dq_date_valid|Is_Fraudulent_Claim|Is_High_Cost|dataset_split|
+---------+----------+-----------+---------------+---------------+------------+------------------+----------+----------------------+--------------------+-----------------+--------------------+------------------+--------------+-------------+-------------------+------------+-------------+
|CLM110013|BENE15435 |PRV51790   |Travel         |TRAVEL         |Settled     |244.07511199775644|200.0     |0.8194198841618822    |0   

# 2.1 Policy churn – scored view

In [ ]:
# Inspect schemas (optional debug)
spark.table("bupa_gold.scored_policy_churn").printSchema()
spark.table("bupa_gold.fact_policies").printSchema()

# Create a richer view by joining
spark.sql("""
CREATE OR REPLACE VIEW bupa_gold.vw_scored_policy_churn AS
SELECT
    fp.Policy_ID,
    fp.Customer_ID,
    fp.Product_Line,
    fp.Channel,
    fp.Sum_Insured_GBP,
    fp.Annual_Premium_GBP,
    fp.Policy_Start_Date,
    fp.Policy_End_Date,
    fp.Policy_Duration_Days,
    fp.Tenure_Band,
    fp.Premium_Band,
    fp.Discount_Band,
    fp.Renewal_Offered_Flag,
    fp.Renewal_Accepted_Flag,
    fp.Renewal_Conversion,
    fp.Renewal_Outcome,
    fp.dq_money_valid,
    fp.dq_discount_valid,
    fp.dq_renewal_valid,
    fp.created_at,
    -- model outputs
    sp.churn_prob,
    sp.churn_prediction
FROM bupa_gold.scored_policy_churn sp
LEFT JOIN bupa_gold.fact_policies fp
  ON sp.Policy_ID = fp.Policy_ID
""")

print("✅ Created rich view: bupa_gold.vw_scored_policy_churn")
spark.table("bupa_gold.vw_scored_policy_churn").show(10, truncate=False)


root
 |-- Policy_ID: string (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- Product_Line: string (nullable = true)
 |-- Channel: string (nullable = true)
 |-- Policy_Start_Date: date (nullable = true)
 |-- Policy_End_Date: date (nullable = true)
 |-- churn_prob: double (nullable = true)
 |-- churn_prediction: double (nullable = true)

root
 |-- Policy_ID: string (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- Product_Line: string (nullable = true)
 |-- Channel: string (nullable = true)
 |-- Sum_Insured_GBP: double (nullable = true)
 |-- Annual_Premium_GBP: double (nullable = true)
 |-- Policy_Start_Date: date (nullable = true)
 |-- Policy_End_Date: date (nullable = true)
 |-- Policy_Duration_Days: integer (nullable = true)
 |-- Renewal_Offered_Flag: integer (nullable = true)
 |-- Renewal_Accepted_Flag: integer (nullable = true)
 |-- Renewal_Conversion: integer (nullable = true)
 |-- Tenure_Band: string (nullable = true)
 |-- Premium_Band: string (nul

25/12/11 16:28:27 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+----------+------------+------------+-------+------------------+------------------+-----------------+---------------+--------------------+-----------+-----------------+-------------+--------------------+---------------------+------------------+---------------+--------------+-----------------+----------------+--------------------------+---------------------+----------------+
|Policy_ID |Customer_ID |Product_Line|Channel|Sum_Insured_GBP   |Annual_Premium_GBP|Policy_Start_Date|Policy_End_Date|Policy_Duration_Days|Tenure_Band|Premium_Band     |Discount_Band|Renewal_Offered_Flag|Renewal_Accepted_Flag|Renewal_Conversion|Renewal_Outcome|dq_money_valid|dq_discount_valid|dq_renewal_valid|created_at                |churn_prob           |churn_prediction|
+----------+------------+------------+-------+------------------+------------------+-----------------+---------------+--------------------+-----------+-----------------+-------------+--------------------+---------------------+------------------

In [ ]:
df= spark.table("bupa_gold.vw_scored_policy_churn")



df.write.format("delta").mode("overwrite").save("abfss://golddata@clientdatastorage.dfs.core.windows.net/vw_scored_policy_churn")

25/12/11 16:32:56 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


In [ ]:
df = spark.read.format("delta").load("abfss://golddata@clientdatastorage.dfs.core.windows.net/vw_scored_policy_churn")

df.show(5, truncate=False)

+----------+------------+------------+-------+-----------------+------------------+-----------------+---------------+--------------------+-----------+-----------------+-------------+--------------------+---------------------+------------------+---------------+--------------+-----------------+----------------+--------------------------+---------------------+----------------+
|Policy_ID |Customer_ID |Product_Line|Channel|Sum_Insured_GBP  |Annual_Premium_GBP|Policy_Start_Date|Policy_End_Date|Policy_Duration_Days|Tenure_Band|Premium_Band     |Discount_Band|Renewal_Offered_Flag|Renewal_Accepted_Flag|Renewal_Conversion|Renewal_Outcome|dq_money_valid|dq_discount_valid|dq_renewal_valid|created_at                |churn_prob           |churn_prediction|
+----------+------------+------------+-------+-----------------+------------------+-----------------+---------------+--------------------+-----------+-----------------+-------------+--------------------+---------------------+------------------+--

In [ ]:
spark.table("bupa_gold.scored_claims_fraud").printSchema()
spark.table("bupa_gold.scored_claims_high_cost").printSchema()


root
 |-- Claim_ID: string (nullable = true)
 |-- Member_Key: string (nullable = true)
 |-- Provider_ID: string (nullable = true)
 |-- Is_Fraudulent_Claim: integer (nullable = true)
 |-- fraud_prob: double (nullable = true)
 |-- prediction: double (nullable = true)
 |-- Claim_Type_Name: string (nullable = true)
 |-- Claim_Status: string (nullable = true)
 |-- Claim_Amount_GBP: double (nullable = true)
 |-- Payout_GBP: double (nullable = true)
 |-- Payout_to_Amount_Ratio: double (nullable = true)
 |-- Provider_Risk_Tier: string (nullable = true)
 |-- dataset_split: string (nullable = true)

root
 |-- Claim_ID: string (nullable = true)
 |-- Member_Key: string (nullable = true)
 |-- Provider_ID: string (nullable = true)
 |-- Is_High_Cost: integer (nullable = true)
 |-- high_cost_prob: double (nullable = true)
 |-- prediction: double (nullable = true)
 |-- Claim_Type_Name: string (nullable = true)
 |-- Claim_Status: string (nullable = true)
 |-- Claim_Amount_GBP: double (nullable = true)
 

# 1️⃣ View for fraud model

In [ ]:
# Cell — View: vw_scored_claims_fraud

spark.sql("""
CREATE OR REPLACE VIEW bupa_gold.vw_scored_claims_fraud AS
SELECT
    Claim_ID,
    Member_Key,
    Provider_ID,
    Claim_Type_Name,
    Claim_Status,
    Claim_Amount_GBP,
    Payout_GBP,
    Payout_to_Amount_Ratio,
    Provider_Risk_Tier,
    -- Model outputs
    fraud_prob,
    prediction        AS fraud_prediction,
    -- Label (present in dev, can be null in prod)
    Is_Fraudulent_Claim,
    dataset_split
FROM bupa_gold.scored_claims_fraud
""")

print("✅ Created view: bupa_gold.vw_scored_claims_fraud")
spark.table("bupa_gold.vw_scored_claims_fraud").show(10, truncate=False)


✅ Created view: bupa_gold.vw_scored_claims_fraud


+---------+----------+-----------+---------------+------------+------------------+----------+----------------------+------------------+-------------------+----------------+-------------------+-------------+
|Claim_ID |Member_Key|Provider_ID|Claim_Type_Name|Claim_Status|Claim_Amount_GBP  |Payout_GBP|Payout_to_Amount_Ratio|Provider_Risk_Tier|fraud_prob         |fraud_prediction|Is_Fraudulent_Claim|dataset_split|
+---------+----------+-----------+---------------+------------+------------------+----------+----------------------+------------------+-------------------+----------------+-------------------+-------------+
|CLM110013|BENE15435 |PRV51790   |Travel         |Settled     |244.07511199775644|200.0     |0.8194198841618822    |Low risk          |0.30197347924196893|0.0             |0                  |train        |
|CLM110020|BENE53030 |PRV53679   |Hospital       |Settled     |4117.6845027538575|2600.0    |0.6314228295686937    |Low risk          |0.30197347924196893|0.0             |

In [ ]:
df= spark.table("bupa_gold.vw_scored_claims_fraud")



df.write.format("delta").mode("overwrite").save("abfss://golddata@clientdatastorage.dfs.core.windows.net/vw_scored_claims_fraud")

In [ ]:
df = spark.read.format("delta").load("abfss://golddata@clientdatastorage.dfs.core.windows.net/vw_scored_claims_fraud")

df.show(5, truncate=False)

+---------+----------+-----------+---------------+------------+------------------+----------+----------------------+------------------+-------------------+----------------+-------------------+-------------+
|Claim_ID |Member_Key|Provider_ID|Claim_Type_Name|Claim_Status|Claim_Amount_GBP  |Payout_GBP|Payout_to_Amount_Ratio|Provider_Risk_Tier|fraud_prob         |fraud_prediction|Is_Fraudulent_Claim|dataset_split|
+---------+----------+-----------+---------------+------------+------------------+----------+----------------------+------------------+-------------------+----------------+-------------------+-------------+
|CLM110013|BENE15435 |PRV51790   |Travel         |Settled     |244.07511199775644|200.0     |0.8194198841618822    |Low risk          |0.30197347924196893|0.0             |0                  |train        |
|CLM110020|BENE53030 |PRV53679   |Hospital       |Settled     |4117.6845027538575|2600.0    |0.6314228295686937    |Low risk          |0.30197347924196893|0.0             |

# 2️⃣ View for high-cost model

In [ ]:
# Cell — View: vw_scored_claims_high_cost

spark.sql("""
CREATE OR REPLACE VIEW bupa_gold.vw_scored_claims_high_cost AS
SELECT
    Claim_ID,
    Member_Key,
    Provider_ID,
    Claim_Type_Name,
    Claim_Status,
    Claim_Amount_GBP,
    Payout_GBP,
    Payout_to_Amount_Ratio,
    Provider_Risk_Tier,
    -- Model outputs
    high_cost_prob,
    prediction        AS high_cost_prediction,
    -- Label (present in dev, can be null in prod)
    Is_High_Cost,
    dataset_split
FROM bupa_gold.scored_claims_high_cost
""")

print("✅ Created view: bupa_gold.vw_scored_claims_high_cost")
spark.table("bupa_gold.vw_scored_claims_high_cost").show(10, truncate=False)


✅ Created view: bupa_gold.vw_scored_claims_high_cost


+---------+----------+-----------+---------------+------------+------------------+----------+----------------------+------------------+--------------------+--------------------+------------+-------------+
|Claim_ID |Member_Key|Provider_ID|Claim_Type_Name|Claim_Status|Claim_Amount_GBP  |Payout_GBP|Payout_to_Amount_Ratio|Provider_Risk_Tier|high_cost_prob      |high_cost_prediction|Is_High_Cost|dataset_split|
+---------+----------+-----------+---------------+------------+------------------+----------+----------------------+------------------+--------------------+--------------------+------------+-------------+
|CLM110013|BENE15435 |PRV51790   |Travel         |Settled     |244.07511199775644|200.0     |0.8194198841618822    |Low risk          |0.021246719556712867|0.0                 |0           |train        |
|CLM110020|BENE53030 |PRV53679   |Hospital       |Settled     |4117.6845027538575|2600.0    |0.6314228295686937    |Low risk          |0.31883079704452943 |0.0                 |1  

In [ ]:
df= spark.table("bupa_gold.vw_scored_claims_high_cost")



df.write.format("delta").mode("overwrite").save("abfss://golddata@clientdatastorage.dfs.core.windows.net/vw_scored_claims_high_cost")

In [ ]:
df = spark.read.format("delta").load("abfss://golddata@clientdatastorage.dfs.core.windows.net/vw_scored_claims_high_cost")

df.show(5, truncate=False)

+---------+----------+-----------+---------------+------------+------------------+----------+----------------------+------------------+--------------------+--------------------+------------+-------------+
|Claim_ID |Member_Key|Provider_ID|Claim_Type_Name|Claim_Status|Claim_Amount_GBP  |Payout_GBP|Payout_to_Amount_Ratio|Provider_Risk_Tier|high_cost_prob      |high_cost_prediction|Is_High_Cost|dataset_split|
+---------+----------+-----------+---------------+------------+------------------+----------+----------------------+------------------+--------------------+--------------------+------------+-------------+
|CLM110013|BENE15435 |PRV51790   |Travel         |Settled     |244.07511199775644|200.0     |0.8194198841618822    |Low risk          |0.021246719556712867|0.0                 |0           |train        |
|CLM110020|BENE53030 |PRV53679   |Hospital       |Settled     |4117.6845027538575|2600.0    |0.6314228295686937    |Low risk          |0.31883079704452943 |0.0                 |1  

In [8]:
spark.table("bupa_gold.scored_policy_churn").printSchema()

root
 |-- Policy_ID: string (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- Product_Line: string (nullable = true)
 |-- Channel: string (nullable = true)
 |-- Policy_Start_Date: date (nullable = true)
 |-- Policy_End_Date: date (nullable = true)
 |-- churn_prob: double (nullable = true)
 |-- churn_prediction: double (nullable = true)



In [9]:
spark.table("bupa_gold.star_policies").printSchema()

root
 |-- Policy_ID: string (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- Product_Line_Code: string (nullable = true)
 |-- Product_Line_Name: string (nullable = true)
 |-- Channel_Code: string (nullable = true)
 |-- Channel_Name: string (nullable = true)
 |-- Sum_Insured_GBP: double (nullable = true)
 |-- Annual_Premium_GBP: double (nullable = true)
 |-- Policy_Start_Date: date (nullable = true)
 |-- Policy_End_Date: date (nullable = true)
 |-- Policy_Duration_Days: integer (nullable = true)
 |-- Renewal_Offered_Flag: integer (nullable = true)
 |-- Renewal_Accepted_Flag: integer (nullable = true)
 |-- Renewal_Conversion: integer (nullable = true)
 |-- Tenure_Band: string (nullable = true)
 |-- Premium_Band: string (nullable = true)
 |-- Discount_Band: string (nullable = true)
 |-- Renewal_Outcome: string (nullable = true)
 |-- dq_money_valid: integer (nullable = true)
 |-- dq_discount_valid: integer (nullable = true)
 |-- dq_renewal_valid: integer (nullable = true)


In [4]:
# Cell — Create view vw_star_policies over star_policies

spark.sql("""
CREATE OR REPLACE VIEW bupa_gold.vw_star_policies AS
SELECT
    Policy_ID,
    Customer_ID,
    -- rename *_Name to generic dimension names
    Product_Line_Name  AS Product_Line,
    Channel_Name       AS Channel,
    Sum_Insured_GBP,
    Annual_Premium_GBP,
    Policy_Start_Date,
    Policy_End_Date,
    Policy_Duration_Days,
    Tenure_Band,
    Premium_Band,
    Discount_Band,
    Renewal_Offered_Flag,
    Renewal_Accepted_Flag,
    Renewal_Conversion,
    Renewal_Outcome,
    dq_money_valid,
    dq_discount_valid,
    dq_renewal_valid
FROM bupa_gold.star_policies
""")

print("✅ Created view: bupa_gold.vw_star_policies")
spark.table("bupa_gold.vw_star_policies").show(5, truncate=False)


✅ Created view: bupa_gold.vw_star_policies


25/12/11 18:53:02 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+----------+------------+------------+-------+------------------+------------------+-----------------+---------------+--------------------+-----------+-----------------+-------------+--------------------+---------------------+------------------+---------------+--------------+-----------------+----------------+
|Policy_ID |Customer_ID |Product_Line|Channel|Sum_Insured_GBP   |Annual_Premium_GBP|Policy_Start_Date|Policy_End_Date|Policy_Duration_Days|Tenure_Band|Premium_Band     |Discount_Band|Renewal_Offered_Flag|Renewal_Accepted_Flag|Renewal_Conversion|Renewal_Outcome|dq_money_valid|dq_discount_valid|dq_renewal_valid|
+----------+------------+------------+-------+------------------+------------------+-----------------+---------------+--------------------+-----------+-----------------+-------------+--------------------+---------------------+------------------+---------------+--------------+-----------------+----------------+
|P_00000005|CUST_0000005|Dental      |Partner|1589495.8186594383

In [14]:
spark.table("bupa_gold.vw_scored_policy_churn").show(5, truncate=False)

+----------+------------+------------+-------+-----------------+---------------+---------------------+----------------+
|Policy_ID |Customer_ID |Product_Line|Channel|Policy_Start_Date|Policy_End_Date|churn_prob           |churn_prediction|
+----------+------------+------------+-------+-----------------+---------------+---------------------+----------------+
|P_00000004|CUST_0000004|Health      |Partner|2023-03-07       |2024-08-07     |0.0029980362208160594|0.0             |
|P_00000009|CUST_0000009|Health      |Partner|2024-06-27       |2025-04-13     |0.0029675061986167294|0.0             |
|P_00000019|CUST_0000019|Health      |Partner|2022-12-24       |2023-09-02     |1.0                  |1.0             |
|P_00000037|CUST_0000037|Health      |Partner|2025-08-23       |2026-05-25     |0.0058714330571299185|0.0             |
|P_00000043|CUST_0000043|Motor       |Partner|2022-04-17       |2023-03-25     |0.0028273730684326708|0.0             |
+----------+------------+------------+--

In [23]:
spark.sql("""
CREATE OR REPLACE VIEW bupa_gold.vw_dash_policy_churn AS
SELECT
    -- Star-schema policy attributes
    p.Policy_ID,
    p.Customer_ID,
    p.Product_Line  AS Product_Line,
    p.Channel       AS Channel,
    p.Sum_Insured_GBP,
    p.Annual_Premium_GBP,
    p.Policy_Start_Date,
    p.Policy_End_Date,
    p.Policy_Duration_Days,
    p.Tenure_Band,
    p.Premium_Band,
    p.Discount_Band,
    p.Renewal_Offered_Flag,
    p.Renewal_Accepted_Flag,
    p.Renewal_Conversion,
    p.Renewal_Outcome,
    p.dq_money_valid,
    p.dq_discount_valid,
    p.dq_renewal_valid,

    -- Model outputs (from scored view)
    s.churn_prob,
    s.churn_prediction

FROM bupa_gold.vw_star_policies      p
LEFT JOIN bupa_gold.vw_scored_policy_churn s
  ON p.Policy_ID = s.Policy_ID
""")

print("✅ Created view: bupa_gold.vw_dash_policy_churn")
spark.table("bupa_gold.vw_dash_policy_churn").show(5, truncate=False)


✅ Created view: bupa_gold.vw_dash_policy_churn


+----------+------------+------------+-------+------------------+------------------+-----------------+---------------+--------------------+-----------+-----------------+-------------+--------------------+---------------------+------------------+---------------+--------------+-----------------+----------------+---------------------+----------------+
|Policy_ID |Customer_ID |Product_Line|Channel|Sum_Insured_GBP   |Annual_Premium_GBP|Policy_Start_Date|Policy_End_Date|Policy_Duration_Days|Tenure_Band|Premium_Band     |Discount_Band|Renewal_Offered_Flag|Renewal_Accepted_Flag|Renewal_Conversion|Renewal_Outcome|dq_money_valid|dq_discount_valid|dq_renewal_valid|churn_prob           |churn_prediction|
+----------+------------+------------+-------+------------------+------------------+-----------------+---------------+--------------------+-----------+-----------------+-------------+--------------------+---------------------+------------------+---------------+--------------+-----------------+----

# dashboard view for fraud claims 

In [26]:
# Cell B1 — Claims fraud dashboard view

spark.sql("""
CREATE OR REPLACE VIEW bupa_gold.vw_dash_claims_fraud AS
SELECT
    Claim_ID,
    Member_Key,
    Provider_ID,
    Claim_Type_Name,
    Claim_Status,
    Claim_Amount_GBP,
    Payout_GBP,
    Payout_to_Amount_Ratio,
    Provider_Risk_Tier,
    -- model outputs
    fraud_prob,
    fraud_prediction,
    -- label & metadata (useful in dev, null in prod scoring)
    Is_Fraudulent_Claim,
    dataset_split
FROM bupa_gold.vw_scored_claims_fraud
""")

print("✅ Created view: bupa_gold.vw_dash_claims_fraud")
spark.table("bupa_gold.vw_dash_claims_fraud").show(10, truncate=False)


✅ Created view: bupa_gold.vw_dash_claims_fraud


+---------+----------+-----------+---------------+------------+------------------+----------+----------------------+------------------+-------------------+----------------+-------------------+-------------+
|Claim_ID |Member_Key|Provider_ID|Claim_Type_Name|Claim_Status|Claim_Amount_GBP  |Payout_GBP|Payout_to_Amount_Ratio|Provider_Risk_Tier|fraud_prob         |fraud_prediction|Is_Fraudulent_Claim|dataset_split|
+---------+----------+-----------+---------------+------------+------------------+----------+----------------------+------------------+-------------------+----------------+-------------------+-------------+
|CLM110013|BENE15435 |PRV51790   |Travel         |Settled     |244.07511199775644|200.0     |0.8194198841618822    |Low risk          |0.30197347924196893|0.0             |0                  |train        |
|CLM110020|BENE53030 |PRV53679   |Hospital       |Settled     |4117.6845027538575|2600.0    |0.6314228295686937    |Low risk          |0.30197347924196893|0.0             |

In [29]:
# Cell A2 — Claims fraud dashboard view

spark.sql("""
CREATE OR REPLACE VIEW bupa_gold.vw_dash_claims_fraud AS
SELECT
    c.Claim_ID,
    c.Member_Key,
    c.Provider_ID,
    c.Claim_Type_Name,
    c.Claim_Status,
    c.Claim_Amount_GBP,
    c.Payout_GBP,
    c.Payout_to_Amount_Ratio,
    c.Provider_Risk_Tier,
    c.High_Cost_Claim_Flag,
    c.Fraud_Label        AS Label_Fraud,
    -- model output
    f.fraud_prob,
    f.fraud_prediction,
    f.dataset_split,
    c.dq_money_valid,
    c.dq_date_valid
FROM bupa_gold.vw_star_claims         c
LEFT JOIN bupa_gold.vw_scored_claims_fraud f
  ON c.Claim_ID = f.Claim_ID
""")

print("✅ Created view: bupa_gold.vw_dash_claims_fraud")
spark.table("bupa_gold.vw_dash_claims_fraud").show(5, truncate=False)


AnalysisException: [TABLE_OR_VIEW_NOT_FOUND] The table or view `bupa_gold`.`vw_star_claims` cannot be found. Verify the spelling and correctness of the schema and catalog.
If you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.
To tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS.; line 21 pos 5;
'CreateViewCommand `spark_catalog`.`bupa_gold`.`vw_dash_claims_fraud`, SELECT
    c.Claim_ID,
    c.Member_Key,
    c.Provider_ID,
    c.Claim_Type_Name,
    c.Claim_Status,
    c.Claim_Amount_GBP,
    c.Payout_GBP,
    c.Payout_to_Amount_Ratio,
    c.Provider_Risk_Tier,
    c.High_Cost_Claim_Flag,
    c.Fraud_Label        AS Label_Fraud,
    -- model output
    f.fraud_prob,
    f.fraud_prediction,
    f.dataset_split,
    c.dq_money_valid,
    c.dq_date_valid
FROM bupa_gold.vw_star_claims         c
LEFT JOIN bupa_gold.vw_scored_claims_fraud f
  ON c.Claim_ID = f.Claim_ID, false, true,

25/12/12 03:10:15 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 946924 ms exceeds timeout 120000 ms
25/12/12 03:10:16 WARN SparkContext: Killing executors is not supported by current scheduler.
25/12/12 03:10:17 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:124)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$

# high cost claims

In [28]:
# Cell B2 — High-cost claims dashboard view

spark.sql("""
CREATE OR REPLACE VIEW bupa_gold.vw_dash_claims_high_cost AS
SELECT
    Claim_ID,
    Member_Key,
    Provider_ID,
    Claim_Type_Name,
    Claim_Status,
    Claim_Amount_GBP,
    Payout_GBP,
    Payout_to_Amount_Ratio,
    Provider_Risk_Tier,
    -- model outputs
    high_cost_prob,
    high_cost_prediction,
    -- label & metadata
    Is_High_Cost,
    dataset_split
FROM bupa_gold.vw_scored_claims_high_cost
""")

print("✅ Created view: bupa_gold.vw_dash_claims_high_cost")
spark.table("bupa_gold.vw_dash_claims_high_cost").show(10, truncate=False)


✅ Created view: bupa_gold.vw_dash_claims_high_cost


+---------+----------+-----------+---------------+------------+------------------+----------+----------------------+------------------+--------------------+--------------------+------------+-------------+
|Claim_ID |Member_Key|Provider_ID|Claim_Type_Name|Claim_Status|Claim_Amount_GBP  |Payout_GBP|Payout_to_Amount_Ratio|Provider_Risk_Tier|high_cost_prob      |high_cost_prediction|Is_High_Cost|dataset_split|
+---------+----------+-----------+---------------+------------+------------------+----------+----------------------+------------------+--------------------+--------------------+------------+-------------+
|CLM110013|BENE15435 |PRV51790   |Travel         |Settled     |244.07511199775644|200.0     |0.8194198841618822    |Low risk          |0.021246719556712867|0.0                 |0           |train        |
|CLM110020|BENE53030 |PRV53679   |Hospital       |Settled     |4117.6845027538575|2600.0    |0.6314228295686937    |Low risk          |0.31883079704452943 |0.0                 |1  

In [ ]:
# Cell A3 — High-cost claims dashboard view

spark.sql("""
CREATE OR REPLACE VIEW bupa_gold.vw_dash_claims_high_cost AS
SELECT
    c.Claim_ID,
    c.Member_Key,
    c.Provider_ID,
    c.Claim_Type_Name,
    c.Claim_Status,
    c.Claim_Amount_GBP,
    c.Payout_GBP,
    c.Payout_to_Amount_Ratio,
    c.Provider_Risk_Tier,
    c.High_Cost_Claim_Flag     AS Label_High_Cost,
    h.high_cost_prob,
    h.high_cost_prediction,
    h.dataset_split,
    c.dq_money_valid,
    c.dq_date_valid
FROM bupa_gold.vw_star_claims            c
LEFT JOIN bupa_gold.vw_scored_claims_high_cost h
  ON c.Claim_ID = h.Claim_ID
""")

print("✅ Created view: bupa_gold.vw_dash_claims_high_cost")
spark.table("bupa_gold.vw_dash_claims_high_cost").show(5, truncate=False)
